<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.1: Procesamiento de Textos y Bag of Words

> **Prof. Heider Sanchez**  

##  Introducción
Este laboratorio tiene como objetivo el análisis y búsqueda de documentos textuales utilizando procesamiento de lenguaje natural (NLP) y una base de datos PostgreSQL. Se trabajará paso a paso desde la extracción de los textos hasta la aplicación búsquedas booleanas.


### Objetivos
- Configurar la tabla en PostgreSQL y carga de datos.
- Desde Python leer los textos desde PostgreSQL.
- Realizar el procesamiento de textos: convertir a minúscula, tokenización, stopwords, stemming y frecuencia de términos.
- Almacenar los Bag of Words en la base de datos en formato JSON.
- Realizar búsquedas de documentos similares a una consulta booleana (conectores AND, OR y AND-NOT).


### Requisitos previos

- Tener instalado PostgreSQL en su computadora (ultima versión)
- Tener instalado las siguientes dependencias en Python:

    ```bash
    pip install psycopg2-binary nltk scikit-learn pandas
    ```

- Opcionalmente descargar los recursos de NLTK:

    ```python
    import nltk
    nltk.download('punkt')
    ```


## 1. (2 puntos) Configurar la tabla en PostgreSQL y carga de datos


### Crear las tablas

Crear la tabla en PostgreSQL para almacenar los textos de noticias y el bag of words:

```sql
CREATE TABLE noticias (
    id SERIAL PRIMARY KEY,
    url TEXT,
    contenido TEXT,
    categoria VARCHAR(50),
    bag_of_words JSONB
);
```

Además, crear una tabla para almacenar los stopwords

```sql
CREATE TABLE stopwords (
    id SERIAL PRIMARY KEY,
    word TEXT UNIQUE NOT NULL
);
```

### Carga de datos en PostgreSQL

Proceder a cargar el dataset de noticias `news_es.csv` y el dataset de stopwords `stoplist_es.txt`.

### Leer desde PostgreSQL con Python

Completar la función para conectarte a PostgreSQL y leer los datos:

In [ ]:
import psycopg2
import pandas as pd

def connect_db():
    conn = psycopg2.connect(
        dbname="",
        user="",
        password="",
        host="",
        port=""
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido FROM noticias;"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

noticias_df = fetch_data()


C:\Users\Familia\AppData\Local\Temp\ipykernel_7276\2713528029.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 2. (4 puntos) Preprocesamiento de texto

Implementar la función `preprocess` que reciba un texto y realice los siguiente:
- Convertir el texto a minuscula.
- Tokenización.
- Eliminación de stopwords
- Stemming (raíz de las palabras)

In [11]:
import nltk
from nltk.stem import SnowballStemmer

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Familia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Familia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:

def load_stopwords():
    conn = connect_db()
    cursor = conn.cursor()
    cursor.execute("SELECT word FROM stopwords;")
    stopwords_set = set(row[0] for row in cursor.fetchall())
    cursor.close()
    conn.close()
    return stopwords_set

stopwords_es = load_stopwords()
stemmer = SnowballStemmer('spanish')

def preprocess(text):
    text = text.lower()

    text = text.replace('¡', '').replace('¿', '')

    raw_tokens = nltk.word_tokenize(text)
    
    filtered_words = [
        token for token in raw_tokens 
        if token.isalnum() and token not in stopwords_es
    ]

    stemmed_words = [stemmer.stem(word) for word in filtered_words]
    
    return stemmed_words

print(preprocess("¡Hola! Esto es una prueba, con números 123 y símbolos #@$%. ¿Funcionará bien?"))


['hol', 'prueb', 'numer', '123', 'simbol', 'funcion']


Luego, implementar una función para calcular la frecuencia de términos:

In [13]:
def compute_bow(text):
    tokens = preprocess(text)
    
    bow = {}
    for token in tokens:
        if token in bow:
            bow[token] += 1
        else:
            bow[token] = 1
            
    return bow

print(compute_bow("Esta es una prueba con prueba y más prueba. ¡Prueba!. Programadores programadores programando."))

{'prueb': 4, 'program': 3}


## 3. (3 puntos) Actualizar la base de datos con los Bag of Words

Guardar el resultado del Bag of Words en la columna `bag_of_words` de la tabla:

In [14]:
import json

def update_bow_in_db(dataframe):
    conn = connect_db()
    cursor = conn.cursor()
    
    try:
        print("Calculando BoW y actualizando la base de datos...")
        
        for index, row in dataframe.iterrows():
            id_noticia = row['id']
            contenido_noticia = row['contenido']
            
            bow_dict = compute_bow(contenido_noticia)
            
            bow_json = json.dumps(bow_dict)
            
            query = "UPDATE noticias SET bag_of_words = %s WHERE id = %s;"
            cursor.execute(query, (bow_json, id_noticia))
            
        conn.commit()
        print("Todos los registros de bag_of_words se actualizaron correctamente!")
        
    except Exception as e:
        conn.rollback()
        print(f"Ocurrió un error al actualizar: {e}")
        
    finally:
        cursor.close()
        conn.close()

update_bow_in_db(noticias_df)

Calculando BoW y actualizando la base de datos...
Todos los registros de bag_of_words se actualizaron correctamente!


In [15]:
def fetch_data_bow():
    conn = connect_db()
    
    query = "SELECT id, bag_of_words FROM noticias LIMIT 10;"
    
    df_bow = pd.read_sql(query, conn)
    
    conn.close()
    
    return df_bow

fetch_data_bow()

C:\Users\Familia\AppData\Local\Temp\ipykernel_7276\4101556451.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bow = pd.read_sql(query, conn)


,id,bag_of_words
0,1,"{'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'..."
1,2,"{'6': 1, 'dat': 1, 'deb': 1, 'inc': 1, 'med': ..."
2,3,"{'15': 1, '20': 1, '21': 1, '42': 1, '50': 1, ..."
3,4,"{'3': 1, '9': 1, '07': 1, '09': 1, '11': 2, '2..."
4,5,"{'17': 1, '23': 1, '31': 1, '34': 1, '50': 2, ..."
5,273,"{'4': 1, '10': 1, '19': 1, '21': 1, '47': 1, '..."
6,98,"{'3': 1, '4': 1, '5': 1, '6': 1, '8': 1, '01':..."
7,12,"{'m': 1, 'u': 1, '70': 1, '100': 1, 'and': 1, ..."
8,13,"{'6': 1, '50': 1, '63': 1, '180': 1, '275': 1,..."
9,14,"{'40': 1, 've': 1, 'bas': 2, 'cas': 1, 'ceo': ..."


## 4. (8 puntos) Consulta booleana con filtrado por keywords

Antes de aplicar el filtrado desde Python, es importante entender cómo funciona la consulta de una clave dentro de una columna JSONB en PostgreSQL. 

### Ejemplo consulta SQL en JSON:

```sql
SELECT * FROM noticias WHERE bag_of_words ? 'keyword';
```

Esta consulta selecciona todos los registros en los que el `bag_of_words` (formato JSONB) contiene una clave igual a `'keyword'`. El operador `?` verifica la existencia de una clave dentro de un JSON.

### Consulta booleana 
Implementar una función que permita parsear una consulta textual con conectores AND, OR y AND-NOT y con ello se aplique el filtro correspondiente directamente desde la base de datos. 


In [ ]:
def apply_boolean_query(query_text):
    conn = connect_db()

    parts = query_text.split()
    
    sql_conditions = []
    
    i = 0

    while i < len(parts):
        token = parts[i]
        
        if token == "AND":
            sql_conditions.append("AND")
        elif token == "OR":
            sql_conditions.append("OR")
        elif token == "AND-NOT":
            sql_conditions.append("AND NOT")
        else:

            stems = preprocess(token)
            if stems:
                term = stems[0]
                sql_conditions.append(f"bag_of_words ? '{term}'")
            else:
                pass
        
        i += 1


    if not sql_conditions:
        print("Consulta vacía o solo contiene stopwords.")
        return pd.DataFrame()

    where_clause = " ".join(sql_conditions)
    final_query = f"SELECT id, contenido FROM noticias WHERE {where_clause};"
    
    print(f"DEBUG SQL: {final_query}") 
    
    try:
        df_results = pd.read_sql(final_query, conn)
        return df_results
    except Exception as e:
        print(f"Error en la ejecución SQL: {e}")
        return pd.DataFrame()
    finally:
        conn.close()

### Pruebas funcionales

Realizar al menos 8 pruebas funcionales con mas de dos keywords de consulta:

In [ ]:
test_queries = [
    "transformación AND sostenible",
    "México OR Perú",
    "economía AND-NOT crisis",
    "tecnología AND futuro",
    "deporte OR cultura",
    "gobierno AND inversión",
    "salud AND-NOT enfermedad",
    "educación AND digital"
]

for q in test_queries:
    print(f"Buscando: {q}")
    res = apply_boolean_query(q)
    
    if not res.empty:
        total = len(res)
        print(f"Encontrados {total} documentos.")
        print("-" * 15)
        
        for index, row in res.head(3).iterrows():
            print(f"ID: {row['id']}")
            print(f"Texto: {row['contenido'][:200]}...") 
            print("." * 10)
            
    else:
        print("Sin resultados.")
    
    print("-" * 30)

Buscando: transformación AND sostenible
DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_words ? 'transform' AND bag_of_words ? 'sosten';
Encontrados 68 documentos.
---------------
ID: 5
Texto: Ayer en Cartagena se dio inicio a la versión número 56 de la Convención Bancaria. Este será el primer encuentro de los banqueros del país con el presidente Gustavo Petro quien estará en la sesión de c...
..........
ID: 26
Texto: La responsable de soluciones para clientes y banca digital de Garanti BBVA, Işıl Akdemir Evlioğlu, repasa los 15 años de la banca móvil y los avances de la sostenibilidad en el mundo financiero.
Los ...
..........
ID: 30
Texto: El presidente de BBVA, Carlos Torres Vila, ha anunciado este miércoles que la entidad aumentará un 50% su objetivo de movilizar financiación sostenible. “La sostenibilidad es una oportunidad de negoci...
..........
------------------------------
Buscando: México OR Perú
DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_words ? '

C:\Users\Familia\AppData\Local\Temp\ipykernel_7276\4018676159.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_results = pd.read_sql(final_query, conn)


Encontrados 257 documentos.
---------------
ID: 12
Texto: BBVA ha presentado BBVA Spark, su propuesta integral de servicios financieros para acompañar a las compañías innovadoras en sus distintas fases de crecimiento. Spark permitirá a estas compañías cubrir...
..........
ID: 39
Texto: La Organización para la Cooperación y el Desarrollo Económico Ocde presentó un documento en el que analiza las consecuencias mundiales que traerá el conflicto entre Rusia y Ucrania el cual causará int...
..........
ID: 65
Texto: La Medicina se encuentra en un punto de transformación sin precedentes. La pandemia ha puesto de relieve la importancia de la digitalización y numerosas tecnologías impactan en este ámbito con solucio...
..........
------------------------------
Buscando: economía AND-NOT crisis
DEBUG SQL: SELECT id, contenido FROM noticias WHERE bag_of_words ? 'econom' AND NOT bag_of_words ? 'crisis';
Encontrados 512 documentos.
---------------
ID: 5
Texto: Ayer en Cartagena se dio inicio a la v

## 5. (3 puntos) Actividad Final
- Medir el tiempo de ejecución de las consultas con diferentes tamaños de datos y optimizar el código según sea necesario.


**Entregable:** informe de los resultados obtenidos en formato PDF.